# 🚀 Huấn Luyện Trợ Lý Ảo Alexa Tiếng Việt với Unsloth & Llama 3.2 3B
**Mục tiêu:** Tinh chỉnh mô hình Llama 3.2 3B Instruct bằng phương pháp QLoRA siêu tốc trên Google Colab GPU T4 miễn phí.
- Thời gian huấn luyện: ~10 - 15 phút.
- Đầu ra: Tệp mô hình  (Q4_K_M ~2GB) để nạp thẳng vào Ollama trên Arch Linux.

### Bước 1: Cài đặt Unsloth & Thư viện huấn luyện siêu tốc

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes

### Bước 2: Nạp mô hình gốc Llama 3.2 3B Instruct (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Cấu hình LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### Bước 3: Tải tập dữ liệu dataset.json lên Colab
Chạy ô dưới đây để chọn file  từ máy tính tải lên Colab.

In [ ]:
from google.colab import files
print("👉 Hãy chọn tệp dataset.json từ thư mục voice-ai/training trên máy bạn:")
uploaded = files.upload()

### Bước 4: Định dạng dữ liệu huấn luyện

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output_text in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output_text) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

from datasets import load_dataset
dataset = load_dataset("json", data_files="dataset.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✅ Đã nạp thành công {len(dataset)} mẫu dữ liệu huấn luyện!")

### Bước 5: Bắt đầu Huấn Luyện (Training ~10 phút)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 80,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### Bước 6: Thử nghiệm câu trả lời của mô hình sau khi train

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Bạn là Alexa, một trợ lý ảo giọng nói tiếng Việt thông minh, tự nhiên trên Arch Linux. Trả lời ngắn gọn 1-2 câu không markdown.",
        "Chào bạn, bạn có thể giúp gì cho tôi?",
        ""
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

### Bước 7: Xuất tệp GGUF Q4_K_M cho Ollama & Tải về máy
Tệp GGUF (~2GB) sẽ được lưu và tự động mở hộp thoại tải về máy.

In [ ]:
model.save_pretrained_gguf("alexa_vn_q4", tokenizer, quantization_method = "q4_k_m")

from google.colab import files
import glob
gguf_files = glob.glob("alexa_vn_q4/*.gguf")
if gguf_files:
    print(f"⬇️ Đang tải tệp {gguf_files[0]} về máy tính của bạn...")
    files.download(gguf_files[0])
else:
    print("Tệp GGUF đã được tạo trong thư mục alexa_vn_q4")